In [ ]:
import sys
import xarray as xr
import numpy as np
from matplotlib import pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature
import geopandas as gpd
from shapely.geometry import mapping
import pandas as pd
import seaborn as sns

import warnings
warnings.filterwarnings('ignore')

In [ ]:
datap = "/Users/ellendyer/Documents/GitHub/Isotopes_F4R/plots/"
datael = "/Volumes/ESA_F4R/era/era5/pressure_levels/"
dataes = "/Volumes/ESA_F4R/era/era5/surface/"
dataep = "/Volumes/ESA_F4R/era/era5/era5_surface/"
band = {'N':[5,12,10,31],'EQ':[-5,5,8,29],'S':[-15,-5,12,31]}
Y1=1990
Y2=2024
B = 'EQ'
lon_bnds, lat_bnds = (band[B][2], band[B][3]), (band[B][1],band[B][0])
time_bnds = (str(Y1)+'-01-01',str(Y2)+'-12-31')

dsS = xr.open_dataset("/Volumes/ESA_F4R/era_test/era5_surface_e_tp_1990.nc")
dsP = xr.open_dataset("/Volumes/ESA_F4R/era_test/era5_surface_sp_1990.nc")
dsL = xr.open_dataset("/Volumes/ESA_F4R/era_test/era5_levels_1990.nc")
dsS['valid_time'] = dsP['valid_time']
ds = xr.merge([dsS,dsL,dsP])
ds['e']=-1000.0*ds['e']
ds['q']=1000.0*ds['q']
ds['sp']=ds['sp']/100.0
ds = ds.rename({'pressure_level':'level','valid_time':'time','latitude':'lat','longitude':'lon'})

ds=ds.sel(lat=slice(5,-5),lon=slice(8,29))
print(ds)


### Calculate equivalent potential temperature

### Calculate moisture flux convergence

In [ ]:
#https://geocat-comp.readthedocs.io/en/v2023.06.0/examples/vimfc.html#calculate-moisture-flux-convergence-mfc
import geocat.comp as gc

delta_pressure_levels = gc.meteorology.delta_pressure(pressure_lev=ds.level, surface_pressure=ds.sp)
print('1')
g = 9.80665 # gravitational acceleration (m s-2)
layer_mass_weighting = delta_pressure_levels / g # Layer Mass Weighting
layer_mass_weighting .attrs["units"] = "kg m-2"
print('2')

mass_weighted_vapor = ds.q * layer_mass_weighting # mass weighted 'q'
print('3')
iq = mass_weighted_vapor.sum(dim="level") # Vertically Integrated Vapor
iq.attrs["units"] = "g m-2"
print('4')

du_dx, du_dy = gc.gradient(ds.u) # (s-1)
dv_dx, dv_dy = gc.gradient(ds.v)
dq_dx, dq_dy = gc.gradient(iq) # (g m-3)

advection = (-ds.u * dq_dx) - (ds.v * dq_dy) # (g s-1 m-2)
convergence = iq * np.add(du_dx, dv_dy) # (g s-1 m-2)

mfc = advection - convergence # moisture flux convergence (g m-2 s-1)